# Operator runtime — perf-optimize harness dashboard

Visualizes the `analysis.json` / `diagnosis.json` a harness run produced
(`performance/agent/{runner,parse,diagnose}.py`). Plotting lives in
`visualize.py`; this notebook just composes it.

**Prereqs:** `pip install matplotlib`, and at least one run under
`performance/agent/runs/<operator>/` (see `performance/agent/README.md`).
The charts read plain JSON, so this notebook runs anywhere (host or
container) — no GPU needed.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
import sys
from pathlib import Path

# Make visualize.py importable whether the notebook runs from the repo
# root or from performance/agent/.
HERE = Path.cwd()
AGENT = HERE if (HERE / 'visualize.py').exists() else HERE / 'performance' / 'agent'
sys.path.insert(0, str(AGENT))
RUNS = AGENT / 'runs'

import visualize as viz
%matplotlib inline

available = [p.name for p in sorted(RUNS.glob('*')) if (p / 'analysis.json').exists()]
print('available runs:', available)

## Full dashboard

Set `RUN` to any operator listed above. The dashboard skips the NVTX-phase
and override-sweep panels for operators that are not yet instrumented.

In [ ]:
RUN = RUNS / 'fmcw_range_processor'   # <- change to any available run
viz.dashboard(RUN);

## Individual charts

Load the run once, then draw any panel on its own.

In [ ]:
analysis, diagnosis = viz.load_run(RUN)
viz.plot_kernel_fraction(analysis);   # is the tick kernel-bound or host-bound?

In [ ]:
viz.plot_latency(analysis);           # cold / median / p90 across the size ladder

In [ ]:
viz.plot_host_api(analysis);          # where host CUDA-API time goes

In [ ]:
viz.plot_phases(analysis);            # compute() NVTX phases (instrumented ops)

In [ ]:
viz.plot_override_sweep(analysis);    # per-param [live] reconfiguration cost

In [ ]:
viz.plot_findings(diagnosis)
print(viz.findings_table(diagnosis))

## Cross-operator comparison

Kernel share of the steady tick across operators — the one-glance
host-bound vs kernel-bound summary.

In [ ]:
viz.compare_kernel_fraction([
    RUNS / 'fmcw_range_processor',
    RUNS / 'cfar_detector',
]);